In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/mahdimaktabdar/chatgpt-classification-dataset/sentence_level_data.csv
/kaggle/input/datasets/mahdimaktabdar/chatgpt-classification-dataset/article_level_data.csv


In [2]:
!python -m spacy download en_core_web_lg


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 4.3 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [3]:
# =========================================
# Classic ML baseline for sentence/article level
# Extended with selected strong classifiers
# =========================================

import os
import re
import warnings
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, MaxAbsScaler, StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    HistGradientBoostingClassifier
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression

from xgboost import XGBClassifier

import spacy

warnings.filterwarnings("ignore")

# =========================================
# 1) Paths
# =========================================

ARTICLE_PATH = "/kaggle/input/datasets/mahdimaktabdar/chatgpt-classification-dataset/article_level_data.csv"
SENTENCE_PATH = "/kaggle/input/datasets/mahdimaktabdar/chatgpt-classification-dataset/sentence_level_data.csv"

print("Article path exists :", os.path.exists(ARTICLE_PATH), ARTICLE_PATH)
print("Sentence path exists:", os.path.exists(SENTENCE_PATH), SENTENCE_PATH)

# =========================================
# 2) Load data
# =========================================

article_df = pd.read_csv(ARTICLE_PATH)
sentence_df = pd.read_csv(SENTENCE_PATH)

print("\nArticle shape:", article_df.shape)
print(article_df.head())
print(article_df.columns.tolist())

print("\nSentence shape:", sentence_df.shape)
print(sentence_df.head())
print(sentence_df.columns.tolist())

# =========================================
# 3) Utility functions
# =========================================

def detect_text_label_columns(df):
    text_candidates = [
        "text",
        "content",
        "article",
        "sentence",
        "body",
        "response",
        "prompt"
    ]

    label_candidates = [
        "label",
        "class",
        "target",
        "y",
        "generated"
    ]

    text_col = None
    label_col = None

    for c in df.columns:
        if c.lower() in text_candidates:
            text_col = c
            break

    for c in df.columns:
        if c.lower() in label_candidates:
            label_col = c
            break

    if text_col is None:
        for c in df.columns:
            if df[c].dtype == "object":
                text_col = c
                break

    if label_col is None:
        for c in df.columns:
            if pd.api.types.is_numeric_dtype(df[c]) and df[c].nunique() <= 10:
                label_col = c
                break

    if text_col is None or label_col is None:
        raise ValueError("Could not detect text/label columns automatically.")

    return text_col, label_col


def clean_text(text):
    if pd.isna(text):
        return ""

    text = str(text)
    text = re.sub(r"[\u200b-\u200f\uFEFF]", " ", text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


def prepare_labels(y):
    y = np.array(y)

    if y.dtype == object:
        le = LabelEncoder()
        y = le.fit_transform(y)
        return y, le

    return y, None


def print_basic_info(df, text_col, label_col, name):
    print(f"\n===== {name} dataset info =====")
    print("Text column :", text_col)
    print("Label column:", label_col)
    print("Shape       :", df.shape)
    print("Null texts  :", df[text_col].isna().sum())
    print("Label dist  :")
    print(df[label_col].value_counts(dropna=False))


# =========================================
# 4) Detect columns
# =========================================

article_text_col, article_label_col = detect_text_label_columns(article_df)
sentence_text_col, sentence_label_col = detect_text_label_columns(sentence_df)

print_basic_info(
    article_df,
    article_text_col,
    article_label_col,
    "Article"
)

print_basic_info(
    sentence_df,
    sentence_text_col,
    sentence_label_col,
    "Sentence"
)

# =========================================
# 5) Clean text
# =========================================

article_df = article_df.copy()
sentence_df = sentence_df.copy()

article_df[article_text_col] = article_df[article_text_col].apply(clean_text)
sentence_df[sentence_text_col] = sentence_df[sentence_text_col].apply(clean_text)

article_df = article_df[
    article_df[article_text_col].str.len() > 0
].reset_index(drop=True)

sentence_df = sentence_df[
    sentence_df[sentence_text_col].str.len() > 0
].reset_index(drop=True)

print("\nAfter cleaning:")
print("Article shape :", article_df.shape)
print("Sentence shape:", sentence_df.shape)

# =========================================
# 6) Load spaCy model
# =========================================

nlp = spacy.load("en_core_web_lg")

print("\nspaCy model loaded.")
print("Vector length:", nlp.vocab.vectors_length)

# =========================================
# 7) Text -> vector
# =========================================

def texts_to_vectors(texts, nlp, batch_size=64):
    vectors = []
    dim = nlp.vocab.vectors_length

    for doc in nlp.pipe(texts.tolist(), batch_size=batch_size):
        vec = doc.vector

        if vec is None or len(vec) == 0:
            vec = np.zeros(dim, dtype=np.float32)
        else:
            vec = vec.astype(np.float32)

        vectors.append(vec)

    return np.vstack(vectors)


print("\nBuilding article vectors ...")
X_article = texts_to_vectors(
    article_df[article_text_col],
    nlp,
    batch_size=64
)

y_article, article_le = prepare_labels(
    article_df[article_label_col].values
)

print("Building sentence vectors ...")
X_sentence = texts_to_vectors(
    sentence_df[sentence_text_col],
    nlp,
    batch_size=64
)

y_sentence, sentence_le = prepare_labels(
    sentence_df[sentence_label_col].values
)

print("\nVector shapes:")
print("X_article :", X_article.shape)
print("X_sentence:", X_sentence.shape)

# =========================================
# 8) Train/test split
# =========================================

X_article_train, X_article_test, y_article_train, y_article_test = train_test_split(
    X_article,
    y_article,
    test_size=0.2,
    random_state=42,
    shuffle=True,
    stratify=y_article
)

X_sentence_train, X_sentence_test, y_sentence_train, y_sentence_test = train_test_split(
    X_sentence,
    y_sentence,
    test_size=0.2,
    random_state=42,
    shuffle=True,
    stratify=y_sentence
)

print("\nTrain/test shapes:")
print("Article train:", X_article_train.shape, "test:", X_article_test.shape)
print("Sentence train:", X_sentence_train.shape, "test:", X_sentence_test.shape)

# =========================================
# 9) Models
# =========================================

def build_models():
    models = {
        "MultinomialNB": Pipeline([
            ("scaler", MinMaxScaler()),
            ("clf", MultinomialNB(
                alpha=1.0,
                fit_prior=True
            ))
        ]),

        "SVM_poly": Pipeline([
            ("scaler", MaxAbsScaler()),
            ("clf", SVC(
                kernel="poly",
                degree=6,
                probability=True,
                random_state=42
            ))
        ]),

        "RandomForest": Pipeline([
            ("clf", RandomForestClassifier(
                n_estimators=200,
                max_depth=10,
                random_state=42,
                n_jobs=-1
            ))
        ]),

        "ExtraTrees": Pipeline([
            ("clf", ExtraTreesClassifier(
                n_estimators=200,
                max_depth=None,
                min_samples_split=2,
                random_state=42,
                n_jobs=-1
            ))
        ]),

        "HistGradientBoosting": Pipeline([
            ("clf", HistGradientBoostingClassifier(
                max_iter=200,
                learning_rate=0.05,
                max_leaf_nodes=31,
                l2_regularization=1.0,
                random_state=42
            ))
        ]),

        "XGBoost": Pipeline([
            ("clf", XGBClassifier(
                n_estimators=300,
                max_depth=4,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                min_child_weight=2,
                reg_alpha=0.1,
                reg_lambda=1.0,
                objective="binary:logistic",
                eval_metric="logloss",
                random_state=42,
                n_jobs=-1,
                tree_method="hist"
            ))
        ]),

        "LogisticRegression": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(
                C=1.0,
                max_iter=2000,
                solver="lbfgs",
                random_state=42
            ))
        ]),

        "KNN": Pipeline([
            ("scaler", MaxAbsScaler()),
            ("clf", KNeighborsClassifier(
                n_neighbors=15,
                metric="euclidean"
            ))
        ]),
    }

    return models


# =========================================
# 10) Evaluation
# =========================================

def get_model_classes(model):
    if hasattr(model, "classes_"):
        return model.classes_

    if hasattr(model, "named_steps"):
        classifier = model.named_steps.get("clf")

        if classifier is not None and hasattr(classifier, "classes_"):
            return classifier.classes_

    return None


def evaluate_model(
    model,
    X_train,
    y_train,
    X_test,
    y_test,
    name="model"
):
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_prob = None

    if hasattr(model, "predict_proba"):
        probs = model.predict_proba(X_test)

        if probs.ndim == 2 and probs.shape[1] == 2:
            classes = get_model_classes(model)

            if classes is not None and 1 in classes:
                positive_index = np.where(classes == 1)[0][0]
                y_prob = probs[:, positive_index]

    elif hasattr(model, "decision_function"):
        scores = model.decision_function(X_test)

        if np.ndim(scores) == 1:
            y_prob = scores

    acc = accuracy_score(y_test, y_pred)

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test,
        y_pred,
        average="binary",
        pos_label=1,
        zero_division=0
    )

    cm = confusion_matrix(y_test, y_pred)

    roc_auc = np.nan

    if y_prob is not None and len(np.unique(y_test)) == 2:
        try:
            roc_auc = roc_auc_score(y_test, y_prob)
        except Exception:
            pass

    print(f"\n{'=' * 70}")
    print(name)
    print(f"{'=' * 70}")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1-score : {f1:.4f}")

    if not np.isnan(roc_auc):
        print(f"ROC-AUC  : {roc_auc:.4f}")

    print("\nConfusion Matrix:")
    print(cm)

    print("\nClassification Report:")
    print(
        classification_report(
            y_test,
            y_pred,
            digits=4,
            zero_division=0
        )
    )

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc,
        "cm": cm,
        "y_true": y_test,
        "y_pred": y_pred,
        "trained_model": model
    }


# =========================================
# 11) Run experiments for both levels
# =========================================

def run_experiment_set(
    level_name,
    X_train,
    y_train,
    X_test,
    y_test
):
    models = build_models()
    results = {}

    for model_name, model in models.items():
        results[model_name] = evaluate_model(
            model=clone(model),
            X_train=X_train,
            y_train=y_train,
            X_test=X_test,
            y_test=y_test,
            name=f"{level_name} - {model_name}"
        )

    return results


sentence_results = run_experiment_set(
    level_name="Sentence level",
    X_train=X_sentence_train,
    y_train=y_sentence_train,
    X_test=X_sentence_test,
    y_test=y_sentence_test
)

article_results = run_experiment_set(
    level_name="Article level",
    X_train=X_article_train,
    y_train=y_article_train,
    X_test=X_article_test,
    y_test=y_article_test
)

# =========================================
# 12) Convert results to dataframe
# =========================================

def results_to_dataframe(results, level):
    rows = []

    for model_name, metrics in results.items():
        rows.append({
            "level": level,
            "model": model_name,
            "accuracy": metrics["accuracy"],
            "precision": metrics["precision"],
            "recall": metrics["recall"],
            "f1": metrics["f1"],
            "roc_auc": metrics["roc_auc"]
        })

    return pd.DataFrame(rows)


sentence_results_df = results_to_dataframe(
    sentence_results,
    "Sentence"
)

article_results_df = results_to_dataframe(
    article_results,
    "Article"
)

all_results_df = pd.concat(
    [
        sentence_results_df,
        article_results_df
    ],
    ignore_index=True
)

all_results_df = all_results_df.sort_values(
    by=["level", "f1"],
    ascending=[True, False]
).reset_index(drop=True)

print("\n===== All Results =====")
print(all_results_df.round(4))

# =========================================
# 13) Comparison table
# =========================================

comparison_df = all_results_df.pivot(
    index="model",
    columns="level",
    values=[
        "accuracy",
        "precision",
        "recall",
        "f1",
        "roc_auc"
    ]
)

print("\n===== Comparison Table =====")
print(comparison_df.round(4))

# =========================================
# 14) Save outputs
# =========================================

all_results_df.to_csv(
    "/kaggle/working/classic_ml_results_extended.csv",
    index=False
)

comparison_df.to_csv(
    "/kaggle/working/classic_ml_comparison_extended.csv"
)

print("\nSaved:")
print("/kaggle/working/classic_ml_results_extended.csv")
print("/kaggle/working/classic_ml_comparison_extended.csv")

# =========================================
# 15) Best models per level
# =========================================

best_sentence = sentence_results_df.sort_values(
    "f1",
    ascending=False
).iloc[0]

best_article = article_results_df.sort_values(
    "f1",
    ascending=False
).iloc[0]

print("\n===== Best by F1 =====")

print("Best sentence-level model:")
print(best_sentence)

print("\nBest article-level model:")
print(best_article)



Article path exists : True /kaggle/input/datasets/mahdimaktabdar/chatgpt-classification-dataset/article_level_data.csv
Sentence path exists: True /kaggle/input/datasets/mahdimaktabdar/chatgpt-classification-dataset/sentence_level_data.csv

Article shape: (1018, 3)
   Unnamed: 0                                            article  class
0           0  NLP is a multidisciplinary field that draws fr...      0
1           1  There are a variety of emerging applications f...      0
2           2  As each new means of communication and social ...      0
3           3  These suggestions include:, Learn about the pu...      0
4           4  In recent years there has been growing concern...      0
['Unnamed: 0', 'article', 'class']

Sentence shape: (7344, 3)
   Unnamed: 0                                           sentence  class
0           0  NLP is a multidisciplinary field that draws fr...      0
1           1  In terms of linguistics, a program must be abl...      0
2           2  Of course 